# Risk Intelligence QA Agent — Testing Notebook

This notebook tests each phase of the Risk QA Agent independently.

## Environment Setup

Run Phase 0 first to load all environment variables and verify your setup.

In [ ]:
# === PHASE 0: Setup — Add project root to sys.path ===
import sys
from pathlib import Path

# Project root is the parent of the notebooks directory
PROJECT_ROOT = Path("/home/lourvens/project/agentic-project/NexlifyCorp").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Project root added to sys.path: {PROJECT_ROOT}")

# === PHASE 0: Environment Setup ===
import os

# Load .env file
from dotenv import load_dotenv
load_dotenv()

# Verify critical env vars
required = ["ANTHROPIC_API_KEY"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Missing env vars: {missing}")

print("✅ Environment loaded")
print(f"   ANTHROPIC_API_KEY: {os.getenv('ANTHROPIC_API_KEY')[:8]}...")

In [ ]:
# === PHASE 0: Config Verification ===
from src.core.config import get_config

cfg = get_config()
print(f"✅ Config loaded")
print(f"   Main model: {cfg.anthropic_model}")
print(f"   Fast model: {cfg.anthropic_fast_model}")
print(f"   Qdrant URL: {cfg.qdrant_url}")
print(f"   Collection: {cfg.qdrant_collection}")

In [ ]:
# === PHASE 0: LLM Verification ===
from src.core.llm import get_llm, get_fast_llm, create_llm, create_fast_llm

# Test main LLM
llm = get_llm()
print(f"✅ Main LLM: {llm.model}")

# Test fast LLM
fast_llm = get_fast_llm()
print(f"✅ Fast LLM: {fast_llm.model}")

---

## Phase 1: Route Node

Tests the Haiku classifier that decides which retrieval path(s) to activate.

**Expected outputs:** `route_key` = `public_only`, `internal_only`, or `both`

In [ ]:
# === PHASE 1: Route Node — Test with different query types ===
from langchain_core.messages import HumanMessage
from src.agents.risk_qa_agent.route_node import route_node

test_queries = [
    ("What does NVIDIA's 10-K say about supply chain risk?", "public_only"),
    ("What is our internal supply chain risk assessment?", "internal_only"),
    ("How does our risk compare to what NVIDIA discloses?", "both"),
    ("Tell me about Taiwan geopolitical risk exposure", "both"),
    ("What are the key risk factors in NexlifyCorp's SEC filings?", "public_only"),
]

for query, expected_route in test_queries:
    state = {"messages": [HumanMessage(content=query)]}
    result = route_node(state)
    route = result["route_key"]
    status = "✅" if route == expected_route else "⚠️"
    print(f"{status} Query: {query[:60]}...")
    print(f"   Expected: {expected_route} | Got: {route}")
    print()

---

## Phase 2: Retrieve Node

Tests the dual-path retriever dispatcher. Requires Qdrant to be running with data ingested.

**Expected outputs:** `retrieved_chunks` list populated from the correct source(s)

In [ ]:
# === PHASE 2: Retrieve Node — Test with route_key ===
from langchain_core.messages import HumanMessage
from src.agents.risk_qa_agent.retrieve_node import retrieve_node

# First set route_key via route node
state = {
    "messages": [HumanMessage(content="What is our supply chain risk?")],
    "route_key": "internal_only"  # Simulate route node output
}

result = retrieve_node(state)
chunks = result.get("retrieved_chunks", [])
print(f"✅ Retrieved {len(chunks)} chunks (route_key=internal_only)")
for i, c in enumerate(chunks[:3], 1):
    print(f"   Chunk {i}: {c.get('document_title', 'unknown')[:50]}...")

print()

# Test with public_only
state2 = {
    "messages": [HumanMessage(content="What risk factors does NVIDIA disclose?")],
    "route_key": "public_only"
}
result2 = retrieve_node(state2)
chunks2 = result2.get("retrieved_chunks", [])
print(f"✅ Retrieved {len(chunks2)} chunks (route_key=public_only)")
for i, c in enumerate(chunks2[:3], 1):
    print(f"   Chunk {i}: {c.get('document_title', 'unknown')[:50]}...")

---

## Phase 3: Reason Node

Tests the two-pass synthesis — analyzes chunks and produces a reasoning trace.

**Expected outputs:** `reasoning_trace` (structured analysis) + `citations` list

In [ ]:
# === PHASE 3: Reason Node — Test with sample chunks ===
from langchain_core.messages import HumanMessage
from src.agents.risk_qa_agent.reason_node import reason_node

sample_chunks = [
    {
        "chunk_index": 0,
        "content": "78% of our advanced packaging (CoWoS) supply flows through TSMC facilities in Taiwan. Risk score: 9.0/10 (CRITICAL). Probability: 45-55% within 24 months.",
        "document_id": "NCR-2025-Q2-001",
        "document_title": "2025-Q2-Internal-Risk-Register.md",
        "source_category": "internal_nexlify",
        "access_level": "INTERNAL",
        "document_date": "2025-05-17"
    },
    {
        "chunk_index": 1,
        "content": "NVIDIA discloses Taiwan as primary manufacturing location for advanced packaging. Customer concentration risk is disclosed at product line level, not individual customer level.",
        "document_id": "NVDA-10K-2024",
        "document_title": "NVIDIA Corporation 10-K 2024",
        "source_category": "public_sec",
        "access_level": "PUBLIC",
        "document_date": "2024-01-31"
    }
]

state = {
    "messages": [
        HumanMessage(content="Compare our Taiwan supply risk to what NVIDIA discloses")
    ],
    "retrieved_chunks": sample_chunks,
    "route_key": "both"
}

result = reason_node(state)
print("✅ Reason node output:")
print(f"   Reasoning trace length: {len(result['reasoning_trace'])} chars")
print(f"   Citations count: {len(result['citations'])}")
print()
print("--- Reasoning Trace ---")
print(result["reasoning_trace"][:500])
print("...")

---

## Phase 4: Generate Node

Tests the final answer generation with citations.

**Expected outputs:** `AIMessage` with inline footnote citations

In [ ]:
# === PHASE 4: Generate Node — Test with reasoning trace ===
from langchain_core.messages import HumanMessage, AIMessage
from src.agents.risk_qa_agent.generate_node import generate_node
from src.agents.risk_qa_agent.state import Citation

sample_citations = [
    Citation(
        index=1,
        document_id="NCR-2025-Q2-001",
        document_title="2025-Q2-Internal-Risk-Register.md",
        source_category="internal_nexlify",
        access_level="INTERNAL",
        document_date="2025-05-17",
        excerpt="78% of our advanced packaging (CoWoS) supply flows through TSMC facilities in Taiwan... Risk score: 9.0/10 (CRITICAL)",
        chunk_content="78% of our advanced packaging (CoWoS) supply flows through TSMC..."
    ),
    Citation(
        index=2,
        document_id="NVDA-10K-2024",
        document_title="NVIDIA Corporation 10-K 2024",
        source_category="public_sec",
        access_level="PUBLIC",
        document_date="2024-01-31",
        excerpt="NVIDIA discloses Taiwan as primary manufacturing location for advanced packaging...",
        chunk_content="NVIDIA discloses Taiwan as primary manufacturing location..."
    )
]

reasoning_trace = """REASONING TRACE
━━━━━━━━━━━━━━
Source 1 [INTERNAL]: NCR-2025-Q2-001 — 2025-Q2-Internal-Risk-Register.md
  → Nexlify's Taiwan CoWoS dependency: 78%
  → Risk score: 9.0/10 (CRITICAL)
  → Probability: 45-55% within 24 months

Source 2 [PUBLIC]: NVDA-10K-2024 — NVIDIA Corporation 10-K 2024
  → NVIDIA discloses Taiwan as primary manufacturing location
  → Customer concentration disclosed at product line level, not customer-specific

CONFLICT DETECTED:
Internal assessment (78% CoWoS dependency, CRITICAL rating) vs public disclosure
(no customer-specific quantification). NVIDIA's public risk factors are aggregated
at product line level."""

state = {
    "messages": [
        HumanMessage(content="Compare our Taiwan supply risk to what NVIDIA discloses")
    ],
    "reasoning_trace": reasoning_trace,
    "citations": sample_citations
}

result = generate_node(state)
answer_msg = result["messages"][-1]
print("✅ Generate node output:")
print(f"   Answer length: {len(answer_msg.content)} chars")
print()
print("--- Generated Answer ---")
print(answer_msg.content[:800])
print("...")

---

## Full Graph Invocation

End-to-end test of the complete Risk QA Agent graph.

**Expected:** Full pipeline from query → route → retrieve → reason → generate

In [ ]:
# === FULL GRAPH: End-to-End Invocation ===
from langchain_core.messages import HumanMessage
from src.agents.risk_qa_agent.graph import get_risk_qa_graph

graph = get_risk_qa_graph()
print(f"✅ Graph loaded: {list(graph.nodes.keys())}")

# Run the full graph
query = "What is our Taiwan supply chain risk exposure?"
print(f"\n📋 Query: {query}")

result = graph.invoke({
    "messages": [HumanMessage(content=query)]
})

print("\n✅ Full graph execution complete")
print(f"   Final message type: {type(result['messages'][-1]).__name__}")
print(f"   Answer length: {len(result['messages'][-1].content)} chars")
print(f"   Route key: {result.get('route_key', 'N/A')}")
print(f"   Chunks retrieved: {len(result.get('retrieved_chunks', []))}")
print(f"   Citations: {len(result.get('citations', []))}")

In [ ]:
# === FULL GRAPH: Display Full Answer ===
print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["messages"][-1].content)
print()
print(f"Route: {result.get('route_key', 'N/A')}")
print(f"Chunks: {len(result.get('retrieved_chunks', []))}")

---

## Test Scenarios

Run multiple test scenarios to verify routing works correctly.

In [ ]:
# === SCENARIO TESTS ===
from src.agents.risk_qa_agent.graph import get_risk_qa_graph

graph = get_risk_qa_graph()

scenarios = [
    {
        "query": "What risk factors does NVIDIA disclose in their 10-K?",
        "expected_route": "public_only",
        "description": "Public SEC filings only"
    },
    {
        "query": "What is our internal risk register showing for Q2 2025?",
        "expected_route": "internal_only",
        "description": "Internal docs only"
    },
    {
        "query": "How does our supply chain risk compare to what NVIDIA discloses?",
        "expected_route": "both",
        "description": "Both paths (comparison query)"
    },
]

for i, scenario in enumerate(scenarios, 1):
    print(f"\n{'='*60}")
    print(f"SCENARIO {i}: {scenario['description']}")
    print(f"Query: {scenario['query']}")
    print(f"Expected route: {scenario['expected_route']}")
    print("=" * 60)

    result = graph.invoke({
        "messages": [HumanMessage(content=scenario["query"])]
    })

    route = result.get("route_key", "N/A")
    chunks = len(result.get("retrieved_chunks", []))
    answer_len = len(result["messages"][-1].content)
    
    status = "✅" if route == scenario["expected_route"] else "⚠️"
    print(f"{status} Route: {route} (expected: {scenario['expected_route']})")
    print(f"   Chunks retrieved: {chunks}")
    print(f"   Answer length: {answer_len} chars")
    print()
    print(result["messages"][-1].content[:300])
    print("...")

---

## Troubleshooting

**Qdrant not running?**
```bash
docker run -d --name nexlify-qdrant -p 6333:6333 -p 6334:6334 \
  -v $(pwd)/data/qdrant:/qdrant/storage qdrant/qdrant
```

**No data in Qdrant?**
```bash
uv run python -m src.ingestion.ingestion_pipeline --help
```

**Run LangGraph dev server:**
```bash
langgraph dev
```

**Run this notebook:**
```bash
uv run jupyter notebook
```